In [5]:
pip install -U bitsandbytes>=0.46.1

In [ ]:

import torch
import pandas as pd
import re
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()


model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
config = AutoConfig.from_pretrained(model_id, trust_remote_code=False)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=False
)

adapter_dir = "/content/final-neuro-symbolic-adapter.zip"
try:
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    print("LoRA Adapters Loaded! Ready for Part 2.")
except Exception as e:
    model = base_model
    print("Using base model.")
model.eval()

In [7]:
import math
import re
import signal

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException("Function call timed out")

def solve_problem(question, max_retries=3, timeout_seconds=60):
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)

    try:
        bt = chr(96) * 3

        history = (
            "<|user|>\n"
            "Write a Python script to solve the following math problem. You MUST store the final numerical answer in a variable named 'result'.\n\n"
            "Question: Find the positive root of x^2 - 5x + 6 = 0.<|end|>\n"
            "<|assistant|>\n"
            f"{bt}python\n"
            "import math\n"
            "# The quadratic formula is (-b + sqrt(b^2 - 4ac)) / 2a\n"
            "a, b, c = 1, -5, 6\n"
            "root1 = (-b + math.sqrt(b**2 - 4*a*c)) / (2*a)\n"
            "root2 = (-b - math.sqrt(b**2 - 4*a*c)) / (2*a)\n"
            "result = max(root1, root2)\n"
            f"{bt}<|end|>\n"
            "<|user|>\n"
            "Question: If 2^x = 32, what is x^3?<|end|>\n"
            "<|assistant|>\n"
            f"{bt}python\n"
            "import math\n"
            "# 2^x = 32 implies x = log2(32)\n"
            "x = math.log2(32)\n"
            "result = x**3\n"
            f"{bt}<|end|>\n"
            "<|user|>\n"
            f"Question: {question}<|end|>\n"
            "<|assistant|>\n"
            f"{bt}python\n"
        )

        last_full_text = "" # Initialize to store the last generated text

        for attempt in range(max_retries):
            inputs = tokenizer(history, return_tensors="pt").to("cuda")
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                    temperature=0.0,
                    pad_token_id=tokenizer.eos_token_id,
                    use_cache=False
                )

            input_len = inputs["input_ids"].shape[1]
            new_text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
            last_full_text = new_text # Store the last generated text

            code = new_text.split(bt)[0].strip()
            if code.startswith("python"):
                code = code[6:].strip()

            if not code or len(code) < 3:
                print(f"   ⚠️ [Attempt {attempt+1}] Model output blank code. Retrying...")
                history += new_text + f"\n{bt}\n<|user|>\nYou wrote an empty script. Write valid Python code.<|end|>\n<|assistant|>\n{bt}python\n"
                continue

            answer, status = execute_with_guardrails(code)

            if status.startswith("Success") and answer is not None:
                print(f"   ✅ [Attempt {attempt+1}] {status}! AI Calculated: {answer}")
                return answer, attempt + 1

            print(f"🛡️ [Agent Guardrail] Caught AI mistake: {status.split(':')[0]}. Forcing self-correction...")
            history += new_text + f"\n{bt}\n<|user|>\nYour code failed with this error:\n{status}\nFix the logic and rewrite the complete code.<|end|>\n<|assistant|>\n{bt}python\n"

        # Failsafe if all retries fail
        print("   -> [Guardrail] Max retries reached. Activating Failsafe Extractor...")
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", last_full_text)
        if numbers:
            return float(numbers[-1]), "Failsafe Regex Extraction"

        return 0.0, "Total Failure"
    except TimeoutException:
        print(f"   -> [Guardrail] Function call timed out after {timeout_seconds} seconds. Returning failure.")
        return 0.0, "Timeout Failure"
    finally:
        signal.alarm(0)

In [8]:
def execute_with_guardrails(code):
    exec_globals = {}
    try:
        exec(code, exec_globals)
        if 'result' in exec_globals:
            return exec_globals['result'], 'Success'
        else:
            return None, 'Failure: Result variable not defined.'
    except Exception as e:
        return None, f'Failure: {type(e).__name__}: {e}'

In [11]:

import pandas as pd

print("\nIsolated Non-Linear Generalization Test...")


df = pd.read_csv('/content/nonlinear_test.csv')


results_log = []
correct = 0

for i, row in df.iterrows():
    if i + 1 == 165:
        continue
    print(f"\n\t")
    print(f"Q{i+1}/{len(df)}: {row['question']}")

    target = str(row['answer']).strip()
    pred, method = solve_problem(row['question'], max_retries = 3, timeout_seconds = 90)

    try:
        is_match = abs(float(pred) - float(target)) < 1e-4
    except:
        is_match = str(pred).strip() == str(target).strip()

    status = "Correct" if is_match else "Wrong"
    if is_match:
        correct += 1
        print(f"RESULT: Correct! (AI: {pred} | Method: {method})")
    else:
        print(f"RESULT: Wrong. (AI: {pred} | Target: {target} | Method: {method})")


    results_log.append({
        "Question"      : row['question'],
        "Target"        : target,
        "AI_Prediction" : pred,
        "Method_Used"   : method,
        "Status"        : status
    })

df_results = pd.DataFrame(results_log)
df_results.to_csv('execution_telemetry.csv', index=False)

accuracy = (correct / len(df)) * 100
print(f"\nExecution Complete! Accuracy: {accuracy:.1f}%")



Isolated Non-Linear Generalization Test...

	
Q1/250: How many vertical asymptotes does the graph of $y=\frac{2}{x^2+x-6}$ have?
   ✅ [Attempt 1] Success! AI Calculated: 2
RESULT: Correct! (AI: 2 | Method: 1)

	
Q2/250: What is the positive difference between $120\%$ of 30 and $130\%$ of 20?
   ✅ [Attempt 1] Success! AI Calculated: 10.0
RESULT: Correct! (AI: 10.0 | Method: 1)

	
Q3/250: If $2^8=4^x$, what is the value of $x$?
   ✅ [Attempt 1] Success! AI Calculated: 4
RESULT: Correct! (AI: 4 | Method: 1)

	
Q4/250: What is the 100th term of the arithmetic sequence 6, 10, 14, 18, ...?
   ✅ [Attempt 1] Success! AI Calculated: 402
RESULT: Correct! (AI: 402 | Method: 1)

	
Q5/250: Mr. Madoff invests 1000 dollars in a fund that compounds annually at a constant interest rate.  After three years, his investment has grown to 1225 dollars.  What is the annual interest rate, as a percentage?  (Round your answer to the nearest integer.)
   -> [Guardrail] Function call timed out after 90 seconds.